# Week 38, Day 6: Project review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w038d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('errors.py', 'w', encoding='utf-8') as f:
    f.write('"""Every error the library raises on purpose."""\n\n\nclass LibraryError(Exception):\n    """Base class for the library\'s own errors."""\n\n\nclass InvalidInput(LibraryError, ValueError):\n    """Something typed or loaded doesn\'t make sense."""\n\n\nclass NotFound(LibraryError, LookupError):\n    """There\'s no book or member with that id."""\n\n\nclass NotAllowed(LibraryError):\n    """The library\'s rules don\'t allow this right now."""\n\n\nclass StorageError(LibraryError):\n    """The library\'s data couldn\'t be read or saved."""\n')
with open('values.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and checking the values people type, and showing money."""\nfrom errors import InvalidInput\n\nKINDS = {"B": "book", "M": "member"}\n\n\ndef clean_isbn(text):\n    """\'978-0-14-118776-1\' as \'9780141187761\'. Raises InvalidInput."""\n    shown = text.strip()\n    digits = shown.replace("-", "").replace(" ", "")\n    if len(digits) != 13 or not (digits.isascii() and digits.isdigit()):\n        raise InvalidInput(f"an ISBN has 13 digits: {shown!r}")\n    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(digits))\n    if total % 10 != 0:\n        raise InvalidInput(f"wrong check digit: {shown!r}")\n    return digits\n\n\ndef clean_text(text, what, longest):\n    """\'  Things   Fall Apart \' as \'Things Fall Apart\'. Raises InvalidInput."""\n    cleaned = " ".join(text.split())\n    if not cleaned:\n        raise InvalidInput(f"{what} is missing")\n    if len(cleaned) > longest:\n        raise InvalidInput(f"{what} is longer than {longest} characters")\n    return cleaned\n\n\ndef parse_id(text, kind):\n    """\' b12 \' as \'B12\', for kind \'B\'. Raises InvalidInput."""\n    cleaned = text.strip().upper()\n    number = cleaned[1:]\n    if not cleaned.startswith(kind) or not (number.isascii() and number.isdigit()) or number.startswith("0"):\n        raise InvalidInput(f"not a {KINDS[kind]} id: {text.strip()!r}")\n    return cleaned\n\n\ndef parse_pence(text):\n    """\'£1.20\' or \'0.6\' as whole pence. Raises InvalidInput."""\n    shown = text.strip()\n    pounds, dot, pence = shown.removeprefix("£").partition(".")\n    if (not (pounds.isascii() and pounds.isdigit())\n            or (dot and not (pence.isascii() and pence.isdigit() and len(pence) <= 2))):\n        raise InvalidInput(f"not an amount: {shown!r}")\n    return int(pounds) * 100 + (int(pence.ljust(2, "0")) if dot else 0)\n\n\ndef format_pence(pence):\n    """60 as \'£0.60\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef check_all(**checks):\n    """Run each check (a function of no arguments); return their results by name, or raise one\n    InvalidInput listing every problem, in order."""\n    results, problems = {}, []\n    for name, check in checks.items():\n        try:\n            results[name] = check()\n        except InvalidInput as err:\n            problems.append(str(err))\n    if problems:\n        raise InvalidInput("; ".join(problems))\n    return results\n')
with open('domain.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s things and rules. No input, output or storage here."""\nfrom dataclasses import dataclass, field\nfrom datetime import date, timedelta\n\nfrom errors import InvalidInput, NotAllowed\nfrom values import format_pence\n\nLOAN_DAYS = 21\nFINE_PER_DAY = 20      # pence for each day late\nMAX_FINE = 500         # the most one loan can cost\nFINE_LIMIT = 500       # owing this much stops a member borrowing\n\n\n@dataclass\nclass Book:\n    book_id: str\n    isbn: str\n    title: str\n    author: str\n    borrower: str | None = None\n    holds: list[str] = field(default_factory=list)\n\n    @property\n    def on_loan(self):\n        return self.borrower is not None\n\n    def describe(self):\n        return f"{self.book_id}: {self.title} by {self.author}"\n\n\n@dataclass\nclass Member:\n    member_id: str\n    name: str\n    max_loans: int = 3\n    owes: int = 0\n\n\n@dataclass\nclass Loan:\n    loan_id: int\n    book_id: str\n    member_id: str\n    out: date\n    due: date\n    returned: date | None = None\n\n    @property\n    def is_open(self):\n        return self.returned is None\n\n    def days_late(self, today):\n        end = self.returned or today\n        return max(0, (end - self.due).days)\n\n    def fine(self, today):\n        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)\n\n\ndef check_can_borrow(member, book, open_loans, today):\n    """Raise NotAllowed with the first reason the member can\'t borrow the book today."""\n    if book.on_loan:\n        raise NotAllowed(f"{book.book_id} is on loan")\n    if book.holds and book.holds[0] != member.member_id:\n        raise NotAllowed(f"{book.book_id} is being kept for someone else")\n    if len(open_loans) >= member.max_loans:\n        raise NotAllowed(f"{member.name} already has {len(open_loans)} books")\n    if any(loan.days_late(today) for loan in open_loans):\n        raise NotAllowed(f"{member.name} has an overdue book")\n    if member.owes >= FINE_LIMIT:\n        raise NotAllowed(f"{member.name} owes {format_pence(member.owes)}")\n\n\ndef lend(book, member, open_loans, today, loan_id):\n    """Lend the book, or raise NotAllowed. Returns the new Loan."""\n    check_can_borrow(member, book, open_loans, today)\n    if book.holds:\n        book.holds.pop(0)\n    book.borrower = member.member_id\n    return Loan(loan_id, book.book_id, member.member_id, today, today + timedelta(days=LOAN_DAYS))\n\n\ndef give_back(book, loan, member, today):\n    """Close the loan and add any fine to what the member owes. Returns the fine."""\n    assert loan.is_open and loan.book_id == book.book_id and loan.member_id == member.member_id\n    loan.returned = today\n    book.borrower = None\n    fine = loan.fine(today)\n    member.owes += fine\n    return fine\n\n\ndef reserve(book, member):\n    """Join the queue for a book. Returns the member\'s place in it."""\n    if not book.on_loan and not book.holds:\n        raise NotAllowed(f"{book.book_id} is on the shelf: borrow it instead")\n    if book.borrower == member.member_id:\n        raise NotAllowed(f"{member.name} already has {book.book_id}")\n    if member.member_id in book.holds:\n        raise NotAllowed(f"{member.name} is already waiting for {book.book_id}")\n    book.holds.append(member.member_id)\n    return len(book.holds)\n\n\ndef pay(member, pence):\n    """Take a payment towards the member\'s fines. Returns what they still owe."""\n    if pence <= 0:\n        raise InvalidInput("a payment must be more than £0")\n    if pence > member.owes:\n        raise NotAllowed(f"{member.name} only owes {format_pence(member.owes)}")\n    member.owes -= pence\n    return member.owes\n')
with open('store.py', 'w', encoding='utf-8') as f:
    f.write('"""Where the library\'s data is kept: in memory, or in a JSON file."""\nimport json\nimport logging\nimport os\nfrom dataclasses import asdict\nfrom datetime import date\n\nfrom domain import Book, Loan, Member\nfrom errors import NotFound, StorageError\n\nlog = logging.getLogger("shelf.store")\n\n\nclass MemoryStore:\n    """Keeps everything in dictionaries. commit() does nothing: there\'s nowhere to save."""\n\n    def __init__(self):\n        self.books = {}\n        self.members = {}\n        self.loans = []\n        self.counters = {"B": 0, "M": 0, "L": 0}\n\n    def next_id(self, kind):\n        """\'B1\', \'B2\', ... for books, \'M1\', ... for members, and 1, 2, ... for loans. Never reused."""\n        self.counters[kind] += 1\n        number = self.counters[kind]\n        return number if kind == "L" else f"{kind}{number}"\n\n    def add_book(self, book):\n        self.books[book.book_id] = book\n\n    def add_member(self, member):\n        self.members[member.member_id] = member\n\n    def add_loan(self, loan):\n        self.loans.append(loan)\n\n    def book(self, book_id):\n        try:\n            return self.books[book_id]\n        except KeyError:\n            raise NotFound(f"no book {book_id}") from None\n\n    def member(self, member_id):\n        try:\n            return self.members[member_id]\n        except KeyError:\n            raise NotFound(f"no member {member_id}") from None\n\n    def open_loans(self, member_id=None):\n        return [loan for loan in self.loans\n                if loan.is_open and (member_id is None or loan.member_id == member_id)]\n\n    def open_loan_for(self, book_id):\n        for loan in self.loans:\n            if loan.is_open and loan.book_id == book_id:\n                return loan\n        raise NotFound(f"{book_id} isn\'t on loan")\n\n    def commit(self):\n        pass\n\n\ndef to_data(store):\n    """Everything in the store as plain dictionaries and lists, ready for JSON."""\n    def loan_data(loan):\n        d = asdict(loan)\n        d["out"], d["due"] = loan.out.isoformat(), loan.due.isoformat()\n        d["returned"] = loan.returned.isoformat() if loan.returned else None\n        return d\n    return {\n        "books": [asdict(b) for b in store.books.values()],\n        "members": [asdict(m) for m in store.members.values()],\n        "loans": [loan_data(loan) for loan in store.loans],\n        "counters": dict(store.counters),\n    }\n\n\ndef fill(store, data):\n    """Load data made by to_data into an empty store."""\n    for d in data["books"]:\n        store.add_book(Book(**d))\n    for d in data["members"]:\n        store.add_member(Member(**d))\n    for d in data["loans"]:\n        d = dict(d)\n        d["out"], d["due"] = date.fromisoformat(d["out"]), date.fromisoformat(d["due"])\n        d["returned"] = date.fromisoformat(d["returned"]) if d["returned"] else None\n        store.add_loan(Loan(**d))\n    store.counters = dict(data["counters"])\n\n\nclass JsonStore(MemoryStore):\n    """A MemoryStore that loads from a JSON file when it\'s made, and saves on commit()."""\n\n    def __init__(self, path):\n        super().__init__()\n        self.path = path\n        self._load()\n\n    def _load(self):\n        try:\n            with open(self.path, encoding="utf-8") as f:\n                data = json.load(f)\n        except FileNotFoundError:\n            log.info("no file at %s yet: starting empty", self.path)\n            return\n        except (OSError, ValueError) as err:\n            raise StorageError(f"can\'t read {self.path}") from err\n        try:\n            fill(self, data)\n        except (KeyError, TypeError, ValueError) as err:\n            raise StorageError(f"{self.path} is damaged") from err\n        log.info("loaded %d books, %d members and %d loans from %s",\n                 len(self.books), len(self.members), len(self.loans), self.path)\n\n    def commit(self):\n        """Save everything safely: write a temporary file, then replace the old file in one step."""\n        temp = self.path + ".tmp"\n        try:\n            with open(temp, "w", encoding="utf-8") as f:\n                json.dump(to_data(self), f, indent=1, ensure_ascii=False)\n            os.replace(temp, self.path)\n        except OSError as err:\n            raise StorageError(f"can\'t save to {self.path}") from err\n        finally:\n            if os.path.exists(temp):\n                os.remove(temp)\n        log.debug("saved %s", self.path)\n')
with open('service.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s use cases. Takes and returns values; never prints."""\nimport logging\n\nfrom domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve\nfrom values import check_all, clean_isbn, clean_text, parse_id, parse_pence\n\nlog = logging.getLogger("shelf.service")\n\n\nclass Library:\n    def __init__(self, store, today):\n        self.store = store\n        self.today = today          # a function returning today\'s date\n\n    def book(self, book_id):\n        return self.store.book(parse_id(book_id, "B"))\n\n    def member(self, member_id):\n        return self.store.member(parse_id(member_id, "M"))\n\n    def add_book(self, isbn, title, author):\n        v = check_all(isbn=lambda: clean_isbn(isbn),\n                      title=lambda: clean_text(title, "title", 100),\n                      author=lambda: clean_text(author, "author", 60))\n        book = Book(self.store.next_id("B"), v["isbn"], v["title"], v["author"])\n        self.store.add_book(book)\n        self.store.commit()\n        log.info("added %s", book.book_id)\n        return book\n\n    def join(self, name):\n        member = Member(self.store.next_id("M"), clean_text(name, "name", 60))\n        self.store.add_member(member)\n        self.store.commit()\n        log.info("added %s", member.member_id)\n        return member\n\n    def lend(self, book_id, member_id):\n        book, member = self.book(book_id), self.member(member_id)\n        open_loans, today = self.store.open_loans(member.member_id), self.today()\n        check_can_borrow(member, book, open_loans, today)    # first, so a refusal uses up no loan id\n        loan = lend(book, member, open_loans, today, self.store.next_id("L"))\n        self.store.add_loan(loan)\n        self.store.commit()\n        log.info("lent %s to %s", book.book_id, member.member_id)\n        return loan\n\n    def give_back(self, book_id):\n        book = self.book(book_id)\n        loan = self.store.open_loan_for(book.book_id)\n        fine = give_back(book, loan, self.store.member(loan.member_id), self.today())\n        self.store.commit()\n        log.info("%s returned, fine %d", book.book_id, fine)\n        return loan, fine\n\n    def reserve(self, book_id, member_id):\n        place = reserve(self.book(book_id), self.member(member_id))\n        self.store.commit()\n        return place\n\n    def pay(self, member_id, amount):\n        left = pay(self.member(member_id), parse_pence(amount))\n        self.store.commit()\n        return left\n\n    def overdue(self):\n        """(loan, days late) for every late loan, most days late first, then by book id."""\n        today = self.today()\n        late = [(loan, loan.days_late(today)) for loan in self.store.open_loans() if loan.days_late(today)]\n        return sorted(late, key=lambda pair: (-pair[1], pair[0].book_id))\n\n    def find(self, words):\n        """Books whose title or author contains every word (in any case), by id number."""\n        wanted = words.lower().split()\n        found = [b for b in self.store.books.values()\n                 if all(w in f"{b.title} {b.author}".lower() for w in wanted)]\n        return sorted(found, key=lambda b: int(b.book_id[1:]))\n\n    def loans(self, member_id):\n        """The member, and their open loans in the order they were made."""\n        member = self.member(member_id)\n        return member, self.store.open_loans(member.member_id)\n')
with open('cli.py', 'w', encoding='utf-8') as f:
    f.write('"""The text interface: turns typed lines into calls, and results into text."""\nimport shlex\n\nfrom errors import LibraryError\nfrom values import format_pence\n\n\ndef do_add(library, isbn, title, author):\n    return f"added {library.add_book(isbn, title, author).describe()}"\n\n\ndef do_join(library, name):\n    member = library.join(name)\n    return f"joined {member.member_id}: {member.name}"\n\n\ndef do_lend(library, book_id, member_id):\n    loan = library.lend(book_id, member_id)\n    return f"{loan.book_id} lent to {library.member(loan.member_id).name}, due {loan.due.isoformat()}"\n\n\ndef do_return(library, book_id):\n    loan, fine = library.give_back(book_id)\n    text = f"{loan.book_id} returned"\n    if fine:\n        text += f" {loan.days_late(loan.returned)} days late: fine {format_pence(fine)}"\n    holds = library.book(loan.book_id).holds\n    if holds:\n        text += f"; keep it for {library.member(holds[0]).name}"\n    return text\n\n\ndef do_reserve(library, book_id, member_id):\n    place = library.reserve(book_id, member_id)\n    book, member = library.book(book_id), library.member(member_id)\n    return f"{member.name} is number {place} in the queue for {book.book_id}"\n\n\ndef do_pay(library, member_id, amount):\n    left = library.pay(member_id, amount)\n    return f"{library.member(member_id).name} now owes {format_pence(left)}"\n\n\ndef do_overdue(library):\n    late = library.overdue()\n    if not late:\n        return "nothing is overdue"\n    return "\\n".join(f"{loan.book_id} {library.book(loan.book_id).title}: "\n                     f"{library.member(loan.member_id).name}, {days} days late" for loan, days in late)\n\n\ndef do_find(library, *words):\n    books = library.find(" ".join(words))\n    if not books:\n        return "no books found"\n    return "\\n".join(b.describe() + (" (on loan)" if b.on_loan else "") for b in books)\n\n\ndef do_loans(library, member_id):\n    member, loans = library.loans(member_id)\n    if not loans:\n        return f"{member.name} has no books"\n    return "\\n".join(f"{loan.book_id} {library.book(loan.book_id).title}, due {loan.due.isoformat()}" for loan in loans)\n\n\nCOMMANDS = {\n    "add": ("ISBN TITLE AUTHOR", 3, do_add),\n    "join": ("NAME", 1, do_join),\n    "lend": ("BOOK MEMBER", 2, do_lend),\n    "return": ("BOOK", 1, do_return),\n    "reserve": ("BOOK MEMBER", 2, do_reserve),\n    "pay": ("MEMBER AMOUNT", 2, do_pay),\n    "overdue": ("", 0, do_overdue),\n    "find": ("WORDS", None, do_find),\n    "loans": ("MEMBER", 1, do_loans),\n}\n\n\ndef handle(line, library):\n    """One typed line in, one piece of text out."""\n    try:\n        words = shlex.split(line)\n    except ValueError as err:\n        return f"error: {err}"\n    if not words:\n        return ""\n    command, args = words[0].lower(), words[1:]\n    if command not in COMMANDS:\n        return f"unknown command: {command}"\n    shape, count, action = COMMANDS[command]\n    if (count is None and not args) or (count is not None and len(args) != count):\n        return f"usage: {command} {shape}".rstrip()\n    try:\n        return action(library, *args)\n    except LibraryError as err:\n        return f"error: {err}"\n')
with open('app.py', 'w', encoding='utf-8') as f:
    f.write('"""The composition root: builds the program from settings, and runs it."""\nimport logging\nimport sys\nfrom datetime import date\n\nfrom cli import handle\nfrom service import Library\nfrom store import JsonStore, MemoryStore\n\n\ndef build(env, today=date.today):\n    """A Library wired up from settings (a dictionary such as os.environ)."""\n    level = env.get("SHELF_LOG", "WARNING").upper()\n    logging.basicConfig(level=getattr(logging, level, logging.WARNING),\n                        format="%(levelname)s %(name)s: %(message)s", stream=sys.stderr)\n    path = env.get("SHELF_DATA", "library.json")\n    store = MemoryStore() if path == ":memory:" else JsonStore(path)\n    return Library(store, today)\n\n\ndef main(lines, env, today=date.today):\n    library = build(env, today)\n    for line in lines:\n        reply = handle(line, library)\n        if reply:\n            print(reply)\n')

### Bug hunt: Review: a queue for every book

This version of `Book` is a plain class. Bo reserves the first book only, but the output shows Bo waiting for the second book too: `['M2'] ['M2']`. Fix the class so each book has its own queue, and the output is `['M2'] []`.

In [ ]:
class Book:
    holds = []

    def __init__(self, book_id, title):
        self.book_id = book_id
        self.title = title

first = Book("B1", "Things Fall Apart")
second = Book("B2", "Arrow of God")
first.holds.append("M2")
print(first.holds, second.holds)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a queue each\"\n    expect_output(\"['M2'] []\")\n    a, b = Book(\"B7\", \"x\"), Book(\"B8\", \"y\")\n    a.holds.append(\"M1\")\n    assert b.holds == [] and \"holds\" not in vars(Book)")

### Bug hunt: Review: an interface that hides bugs

This `handle` "never crashes": it catches every exception. But a bug in the service (here, a `TypeError`) reaches the volunteer as a puzzling message, and nobody ever sees the traceback that would show where it is. Fix `handle` so it turns only `LibraryError` into a message and lets everything else through. The program should then print `error: no book B9`, and the bug should crash it with a `TypeError`.

In [ ]:
class LibraryError(Exception):
    pass

class Library:
    def lend(self, book_id, member_id):
        if book_id == "B9":
            raise LibraryError("no book B9")
        return len(5)              # a bug

def handle(line, library):
    command, *args = line.split()
    try:
        return f"lent: {library.lend(*args)}"
    except Exception as err:
        return f"error: {err}"

print(handle("lend B9 M1", Library()))
print(handle("lend B1 M1", Library()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only LibraryError\"\n    assert output.startswith(\"error: no book B9\\n\"), output\n    assert error is not None and error[\"type\"] == \"TypeError\", \"the bug should crash the program\"")

### Bug hunt: Review: a store that forgives too much

This store "never fails to start": if the data file can't be loaded, it starts with an empty library. Think about what the next `commit` does to the file. Fix `_load` so a damaged file raises `StorageError("PATH is damaged")` chained from the original error, while a missing file still means an empty library.

In [ ]:
import json

class StorageError(Exception):
    pass

class JsonStore:
    def __init__(self, path):
        self.path = path
        self.books = {}
        self._load()

    def _load(self):
        try:
            with open(self.path, encoding="utf-8") as f:
                self.books = json.load(f)["books"]
        except Exception:
            self.books = {}

    def commit(self):
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump({"books": self.books}, f)

with open("library.json", "w") as f:
    f.write('{"books": {"B1": "Things Fall Apart"}')      # a closing brace lost
try:
    store = JsonStore("library.json")
    store.books["B2"] = "Arrow of God"
    store.commit()
except StorageError as err:
    print("refused:", err)
with open("library.json") as f:
    print(f.read())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"damaged files are refused\"\n    expect_output('refused: library.json is damaged\\n{\"books\": {\"B1\": \"Things Fall Apart\"}')\n    write_file(\"other.json\", \"[1, 2\")\n    try:\n        JsonStore(\"other.json\")\n        assert False\n    except StorageError as err:\n        assert str(err) == \"other.json is damaged\" and err.__cause__ is not None, \"chain it from the original\"\n    assert JsonStore(\"missing.json\").books == {}, \"a missing file is a new library\"\n    write_file(\"ok.json\", '{\"books\": {\"B3\": \"x\"}}')\n    assert JsonStore(\"ok.json\").books == {\"B3\": \"x\"}")

### Bug hunt: Review: a question that changes the answer

`overdue` is a query: it should change nothing. But it sorts the store's list of loans **in place**, so asking "what's overdue?" rearranges the library's history (watch the loan numbers in the last line). Fix it so the output ends with `[1, 2, 3]`, the order the loans were made.

In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass
class Loan:
    loan_id: int
    book_id: str
    due: date

class Store:
    def __init__(self):
        self.loans = [Loan(1, "B4", date(2026, 11, 20)), Loan(2, "B1", date(2026, 11, 3)), Loan(3, "B2", date(2026, 12, 1))]

def overdue(store, today):
    store.loans.sort(key=lambda loan: loan.due)
    return [(loan.book_id, (today - loan.due).days) for loan in store.loans if loan.due < today]

store = Store()
print(overdue(store, date(2026, 11, 25)))
print([loan.loan_id for loan in store.loans])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a query changes nothing\"\n    expect_output(\"[('B1', 22), ('B4', 5)]\\n[1, 2, 3]\")")

### Exercise: Renew a loan

`RenewableLoan` (in the editor) is a `Loan` that also counts its renewals. Write `renew(book, loan, today)`, which renews the loan and returns its new due date. It raises `NotAllowed`, checking in this order:

1. `"B1 is overdue: return it first"` if the loan is late today;
2. `"someone is waiting for B1"` if the book has holds;
3. `"B1 has been renewed 2 times already"` if it's been renewed `MAX_RENEWALS` times.

Otherwise it moves `due` on by `LOAN_DAYS` days **from the current due date** (renewing early doesn't lose days), adds 1 to `renewals`, and returns the new due date. Like `give_back`, it `assert`s that the loan is open and is for this book.

In [ ]:
from dataclasses import dataclass
from datetime import date, timedelta

from domain import LOAN_DAYS, Book, Loan
from errors import NotAllowed

MAX_RENEWALS = 2


@dataclass
class RenewableLoan(Loan):
    renewals: int = 0


def renew(book, loan, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("D = date\ndef make(due=D(2026, 11, 23), renewals=0):\n    return Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\", borrower=\"M1\"), RenewableLoan(1, \"B1\", \"M1\", D(2026, 11, 2), due, renewals=renewals)\ndef refused(book, loan, today, message):\n    try:\n        renew(book, loan, today)\n    except NotAllowed as err:\n        assert str(err) == message, f\"got {str(err)!r}\"\n        return\n    raise AssertionError(f\"should raise NotAllowed: {message}\")\ndef test_renew():\n    \"renewing\"\n    book, loan = make()\n    assert renew(book, loan, D(2026, 11, 10)) == D(2026, 12, 14) and loan.due == D(2026, 12, 14) and loan.renewals == 1\n    assert renew(book, loan, D(2026, 12, 14)) == D(2027, 1, 4) and loan.renewals == 2\n    refused(book, loan, D(2026, 12, 20), \"B1 has been renewed 2 times already\")\ndef test_refused():\n    \"refusals, in order\"\n    book, loan = make()\n    refused(book, loan, D(2026, 11, 24), \"B1 is overdue: return it first\")\n    book.holds = [\"M2\"]\n    refused(book, loan, D(2026, 11, 24), \"B1 is overdue: return it first\")\n    refused(book, loan, D(2026, 11, 20), \"someone is waiting for B1\")\n    book, loan = make(renewals=2)\n    book.holds = [\"M2\"]\n    refused(book, loan, D(2026, 11, 20), \"someone is waiting for B1\")\n    assert loan.due == D(2026, 11, 23) and loan.renewals == 2, \"a refusal changes nothing\"\ndef test_bug():\n    \"a closed loan is a bug\"\n    book, loan = make()\n    loan.returned = D(2026, 11, 20)\n    try:\n        renew(book, loan, D(2026, 11, 21))\n        assert False\n    except AssertionError as err:\n        assert \"should\" not in str(err)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## What you built

| Module | Lines | Its job | Phase 3 ideas it uses |
|---|---|---|---|
| `errors.py` | about 20 | the exception family | exception hierarchies, inheriting from built-ins (Weeks 30 and 37) |
| `values.py` | about 70 | read and check typed values; show money | generous at the edge, strict inside; collecting errors (Week 37) |
| `domain.py` | about 120 | books, members, loans and the rules | dataclasses and properties (Weeks 29 and 32), contracts and `assert` (Week 37) |
| `store.py` | about 120 | memory and JSON storage | repositories, inheritance, translation of errors, atomic saves, logging (Weeks 30, 36 and 37) |
| `service.py` | about 85 | the use cases | the application layer, an injected clock (Week 36) |
| `cli.py` | about 100 | lines in, replies out | a table of command functions (Week 35), catching only `LibraryError` |
| `app.py` | about 25 | settings in, a program out | the composition root and configuration (Week 36) |
| tests | about 150 | domain and service | fixtures, fakes, broken versions (Weeks 24 and 34) |

Here's all of it, as it stands after yesterday:

## A review checklist

Go through this list for your own `shelf`, honestly.

**Does it meet the brief?** Day 1 listed seven things it must do and five rules. Try each one. Does a book with a queue really go only to the first person? Does a member with exactly £5 owing get refused?

**Is it correct at the edges?** A loan returned on its due date, a fine past the cap, a member at their limit, the first and last in a queue, an empty library, `B10` after `B9`.

**Is it safe?** Can any command lose data? What happens with a damaged data file, a full disk, or a crash halfway through a save? Can a refused command leave anything half-changed?

**Are the layers kept?** Does the domain import anything from the store, service or interface? (It mustn't.) Does anything except `cli.py` produce text for people, or anything except `app.py` choose the store?

**Are errors in the right places?** Is every refusal a `LibraryError` with a message a volunteer can act on? Does a bug still crash with a traceback? Is each problem logged once?

**Is it tested, and clear?** Would the tests catch the mistakes you'd most fear? Could someone new find where the fine per day is set, in under a minute?

## The decisions that paid off

- **One exception family.** The interface catches one class, every refusal reaches the volunteer as one line, and bugs are never disguised.
- **A domain with no input, output or storage.** Every rule was tested with plain objects; the store and interface were written later without touching it.
- **Check everything, then change.** `lend` refuses before it changes anything, and the service checks before taking a loan id, so a refusal never leaves a trace.
- **Inject the outside world.** `today`, the store and the settings are passed in, so tests choose the date and use memory instead of files.
- **One place for each decision.** The fine per day is in one constant, the rules in one function, the wiring in one module, and the commands in one table.
- **Tests before refactoring.** Yesterday's refactor changed every line of `cli.py`'s logic, and a long session proved nothing a volunteer sees had changed.

## Reviewing code

Each of these works on a quick try, and each has a mistake that's easy to make in a program like `shelf`. Find it and fix it.

## One more feature: renewals

Members often ask to keep a book longer. The library's rule: a loan can be **renewed** (given another 21 days from its current due date) at most twice, but not if it's already overdue, and not if someone is waiting for the book. Where does that belong? It's a rule about loans and books, so it goes in the **domain**, as a function like `lend`. The service and interface would each need only a few lines, as Step 28's `loans` command did.

In your own `shelf`, `renewals` would go straight into `Loan` (with a default of 0, so old data files still load), a `renew(book_id)` use case into the service, and a `do_renew` function and one table entry into the interface. Notice what **doesn't** change: the store, the composition root, and every other rule.

## Where it could go next

- **A database** instead of a JSON file, once the library has thousands of loans. Only `store.py` would change: a `SqliteStore` with the same methods.
- **A web page** for members to search and reserve from home. A new presentation layer beside `cli.py`; the service is shared.
- **Reminders**: a daily job that finds loans due in two days and sends messages, with the sender injected like the clock.
- **Faster searches**: `find` checks every book, which is fine for a few hundred. For a million, an index from words to books would help; Phase 4 is about exactly that kind of data structure.

## Project quiz

## Week 38 review

Tick off each one honestly. If one feels shaky, go back and redo that day's steps.

- I can turn a brief into a plan of layers, modules and classes, and start with the exception family. ([Day 1](w038d1.html))
- I can check typed values at the edge and collect every problem before refusing. ([Day 1](w038d1.html))
- I can model a domain with dataclasses and write rules that check everything before changing anything. ([Day 2](w038d2.html))
- I can write a store that finds by id, saves atomically, translates errors and logs. ([Day 3](w038d3.html))
- I can write a service, a thin interface and a composition root, and keep the layers apart. ([Day 4](w038d4.html))
- I can test each layer in the way that suits it, and refactor safely with a characterisation test. ([Day 5](w038d5.html))
- I reviewed the project, fixed four typical mistakes, and added renewals in the right layer. (Today)

Tomorrow is a **rest day**. Next week: the **Phase 3 exam**: three days of review (classes; types and design; patterns), a practice exam, the exam itself, and a debrief.